*[cell 0]*

# EMST sample description

The analysed sample of one stamped LVL_0 handoff (pinned as `LVL0_HANDOFF` in `[cell 1]`), described from the analysis
dataset (`dset/all_subjects_long.csv`) and a ledger of the records the dataset does not carry (`[cell 2]`: why a
participant has fewer than two sessions, withdrawals, the corrections the dataset build applied). The intake filters
(`max_age`, sessions, conditions) are read from the handoff's LVL_0 run log; the notebook stops unless the filtered
dataset has exactly the handoff's participants and sessions. Run it after the LVL0 notebook.

**Conditions.** The manuscript calls the two conditions cohorts: T2 = Cohort A (CA), T3 = Cohort B (CB). The code,
file names, exports and the final report keep T2 and T3; only `make_replication_figures.ipynb` prints CA and CB. In
this notebook, "cohort" is the age group (adolescents, 10-17 years; adults, 18 and over); the manuscript's cohorts CA
and CB are the conditions T2 and T3.

| Export (`LVL_OUTS/<handoff stem>_demographics_<key>.csv`) | Content |
|---|---|
| `ledger` | the ledger |
| `flow`, `single_session`, `single_session_counts` | dataset → age exclusion → analysed sample; single-session participants and their reasons |
| `balance` | Table 0.1: sample by condition, T2 vs T3 tests |
| `table` | sample characteristics by cohort, time point and sex (age, depression, anxiety, interval between sessions) |
| `intervals` | same-day and different-day completion, days apart, sessions more than 7 days apart |
| `out_of_range` | responses outside each scale's valid range |

Manuscript counts with no record on this machine (enrolment, recruitment, the pilot, the task version per session) are
printed as flags in `[cell 8]` and never computed.

In [1]:
# [cell 1]
# ===== INPUTS: set these before running =====
LVL0_HANDOFF = "lv0_out_20261106_180438.csv"   # the LVL_0 handoff whose sample this notebook describes, in LVL_OUTS

# =============================================================================
# IMPORTS, INPUT, PATHS
# =============================================================================
import hashlib, json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

# ---- locations: everything is inside /home/sjsuss626/EMST_files/sjs_dissertation ----------------
DSET_PATH = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dset/all_subjects_long.csv")   # the analysis dataset
LVL_OUTS = Path("/home/sjsuss626/EMST_files/sjs_dissertation/dervatives/dynamic_qc/lvl_outs")   # the handoff; this notebook's tables

if LVL0_HANDOFF is None:
    raise SystemExit("LVL0_HANDOFF is not set (top of this cell). Name one LVL_0 handoff; "
                     f"available in {LVL_OUTS}: {sorted(p.name for p in LVL_OUTS.glob('lv0_out_*.csv'))}")
LV0_PATH = LVL_OUTS / LVL0_HANDOFF
_m0 = re.match(r"lv0_out_(\d{8}_\d{6})", LV0_PATH.name)
if not _m0 or LV0_PATH.suffix != ".csv" or not LV0_PATH.is_file():
    raise SystemExit(f"LVL0_HANDOFF = {LVL0_HANDOFF!r} is not an existing LVL_0 handoff "
                     "(lv0_out_<YYYYmmdd_HHMMSS>[...].csv)")
_recs = sorted(LV0_PATH.parent.glob(f"lv0_out_{_m0.group(1)}*_parameters.json"))
if len(_recs) != 1:
    raise SystemExit(f"expected exactly one LVL_0 run log lv0_out_{_m0.group(1)}*_parameters.json beside "
                     f"{LV0_PATH.name}, found {[p.name for p in _recs]}")
LVL0_RUN_LOG = json.load(open(_recs[0]))
if Path(LVL0_RUN_LOG["input_long_path"]).name != DSET_PATH.name:
    raise SystemExit(f"{LV0_PATH.name} was built from {LVL0_RUN_LOG['input_long_path']}, not {DSET_PATH}")
_sha = hashlib.sha256()
with open(DSET_PATH, "rb") as _f:
    for _b in iter(lambda: _f.read(1 << 20), b""):
        _sha.update(_b)
if LVL0_RUN_LOG["outputs"].get("input_long_sha256") != _sha.hexdigest():
    raise SystemExit(f"{DSET_PATH.name} is not the file {LV0_PATH.name} was built from (its run log records "
                     f"{LVL0_RUN_LOG['outputs'].get('input_long_sha256')}); re-run LVL_0 with the current code")
P0 = LVL0_RUN_LOG["parameters"]
for _k in ("max_age", "sessions", "conditions"):
    if _k not in P0:
        raise SystemExit(f"the LVL_0 run log {_recs[0].name} does not record {_k!r}: re-run LVL_0 with the current code")
OUT_STEM = LVL_OUTS / f"{LV0_PATH.stem}_demographics"
ADOLESCENT_MAX_AGE = 17     # cohort: adolescents are 10-17 years old, adults 18 and over
TABLES = {}


def export(key, frame, title):
    """Write one table to LVL_OUTS as <handoff stem>_demographics_<key>.csv and show it."""
    TABLES[key] = frame
    frame.to_csv(f"{OUT_STEM}_{key}.csv", index=False)
    print(title)
    display(frame)


print(f"dataset       : {DSET_PATH}")
print(f"LVL_0 handoff : {LV0_PATH.name}  (run log {_recs[0].name}: max_age {P0['max_age']}, sessions {P0['sessions']}, "
      f"conditions {P0['conditions']})")
print(f"outputs       : {OUT_STEM}_*.csv")

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))


dataset       : /home/sjsuss626/EMST_files/sjs_dissertation/dset/all_subjects_long.csv
LVL_0 handoff : lv0_out_20261106_180438.csv  (run log lv0_out_20261106_180438_parameters.json: max_age 30, sessions [1, 2], conditions ['T2', 'T3'])
outputs       : /home/sjsuss626/EMST_files/sjs_dissertation/dervatives/dynamic_qc/lvl_outs/lv0_out_20261106_180438_demographics_*.csv
interpreter: /usr/bin/python3 (Python 3.10.12)
packages: numpy 1.26.4, pandas 2.2.1, scipy 1.12.0, scikit-learn 1.7.2, statsmodels 0.14.6, hmmlearn 0.3.3, ruptures 1.1.10, joblib 1.5.3, matplotlib 3.10.8


In [2]:
# [cell 2]
# =============================================================================
# LEDGER: RECORDS THE DATASET DOES NOT CARRY
# =============================================================================
# Why a participant has fewer than two usable sessions. Source: build_emst_dataset.ipynb, configuration of the build
# (SESSION_MISSING_REASONS; study notes confirmed 2026-08-09), copied in audit_history/references/. The build detects
# incomplete trial counts, sessions run under the wrong condition and missing files; the category is recorded by hand.
SESSION_MISSING_REASONS = {
    # --- adults ---------------------------------------------------------------
    1014: ("incomplete session", "S1 stopped after 184/200 trials; no documentation of why"),
    1029: ("unknown", "S2 never collected; no documentation identified"),
    1063: ("wrong randomization", "S2 run as T2 on a T3 ID"),
    1067: ("technical interruption", "laptop began an automatic system update during S1"),
    1071: ("unknown", "S1 absent; unconfirmed, no record found"),
    1076: ("procedural error", "S1 administered a second time in place of S2"),
    1078: ("procedural error", "S1 administered a second time in place of S2"),
    2057: ("wrong randomization", "only session (S2) run as T2 on a T3 ID; no usable session"),
    2078: ("did not return", "not enrolled through SONA and did not return for S2"),
    2095: ("procedural error", "S1 administered a second time in place of S2"),
    2105: ("did not return", "participant did not return for S2"),
    2059: ("pilot format", "blocked-design pilot run (no threat scale); both sessions excluded"),
    # --- adolescents ----------------------------------------------------------
    3032: ("wrong randomization", "S1 run as T3 on a T2 ID"),
    3046: ("incomplete session", "S2 stopped after 17/200 trials; RA noted a problem with the S2 code"),
    4019: ("wrong randomization", "S1 run as T2 on a T3 ID"),
    4046: ("unknown", "S1 absent; unconfirmed, no record found"),
}
# Participants with no data in the dataset. Source: the build's AGE_CORRECTIONS notes ("not in dset (withdrew)").
WITHDRAWN = {2106: "withdrew before the rating task"}
# Corrections the build applied to the dataset (already in it; listed so the sample description can cite them).
# Source: build_emst_dataset.ipynb, "Data decisions applied here".
CORRECTIONS = [
    ("age", "1067 -> 37, 1076 -> 56, 1079 -> 25, 2114 -> 41, 2116 -> 56, 3010 -> 14, 3023 -> 12",
     "placeholder ages entered to pass the task's age screening, and cross-session disagreements"),
    ("sex", "1014, 1016, 1020, 1073, 1080, 3004, 3041 from the recorded gender; 3037 male",
     "sex is used, except when sex and gender are opposite binary values (then gender is used as sex)"),
    ("gender", "3037 male; 1059 male", "cross-session disagreements resolved to one value"),
    ("stimuli", "10854 (all), 10862 (T2) dropped", "replaced stimuli"),
]
# Manuscript counts with no record on this machine (enrolment, recruitment, pilot, task version). They are listed as
# flags in [cell 8], never computed.
UNSUPPORTED = [
    ("enrolled: adults 280, adolescents 120 (400)", "enrolment records"),
    ("adult males compensated monetarily: 31", "compensation records"),
    ("adults in the pilot task: 59", "pilot records"),
    ("sessions that ought to have been collected: 600", "enrolment records"),
    ("participants on the original task version: 264 (93.6%)", "session dates (the *_w_rts.tsv file names)"),
]
export("ledger", pd.DataFrame(
    [{"record": "missing session", "id": i, "category": c, "description": d} for i, (c, d) in SESSION_MISSING_REASONS.items()]
    + [{"record": "withdrawal", "id": i, "category": "withdrew", "description": d} for i, d in WITHDRAWN.items()]
    + [{"record": f"correction: {k}", "id": np.nan, "category": v, "description": d} for k, v, d in CORRECTIONS]),
    "Ledger: records behind the sample description")

Ledger: records behind the sample description


,record,id,category,description
0,missing session,1014.0,incomplete session,S1 stopped after 184/200 trials; no documentat...
1,missing session,1029.0,unknown,S2 never collected; no documentation identified
2,missing session,1063.0,wrong randomization,S2 run as T2 on a T3 ID
3,missing session,1067.0,technical interruption,laptop began an automatic system update during S1
4,missing session,1071.0,unknown,"S1 absent; unconfirmed, no record found"
5,missing session,1076.0,procedural error,S1 administered a second time in place of S2
6,missing session,1078.0,procedural error,S1 administered a second time in place of S2
7,missing session,2057.0,wrong randomization,only session (S2) run as T2 on a T3 ID; no usa...
8,missing session,2078.0,did not return,not enrolled through SONA and did not return f...
9,missing session,2095.0,procedural error,S1 administered a second time in place of S2


In [3]:
# [cell 3]
# =============================================================================
# LOAD: THE DATASET, THE INTAKE FILTERS, THE ANALYSED SAMPLE
# =============================================================================
_cols = ["id", "age", "sex", "gender", "condition", "session", "scale", "response", "S1_S2_hours", "same_day",
         "anx_t", "dep_t"]
dset = pd.read_csv(DSET_PATH, usecols=_cols, low_memory=False)
_age_ok = dset["age"] <= P0["max_age"]
analysed = dset[_age_ok & dset["session"].isin(P0["sessions"]) & dset["condition"].isin(P0["conditions"])]
# the analysed sample is the LVL_0 handoff's: same participants, same sessions
_h = pd.read_csv(LV0_PATH, usecols=["id", "session"]).drop_duplicates()
_a = analysed[["id", "session"]].drop_duplicates()
if set(map(tuple, _h.values)) != set(map(tuple, _a.values)):
    raise SystemExit(f"the dataset after the LVL_0 intake filters ({_a.id.nunique()} participants, {len(_a)} sessions) is "
                     f"not the handoff's ({_h.id.nunique()} participants, {len(_h)} sessions)")
cohort = lambda age: np.where(age <= ADOLESCENT_MAX_AGE, "adolescent", "adult")
subj = (analysed.drop_duplicates("id")[["id", "age", "sex", "gender", "condition", "anx_t", "dep_t", "S1_S2_hours",
                                        "same_day"]].reset_index(drop=True))
subj["cohort"] = cohort(subj["age"])
subj["n_sessions"] = subj["id"].map(analysed.drop_duplicates(["id", "session"]).groupby("id").size())
print(f"analysed sample: {len(subj)} participants ({(subj.cohort == 'adolescent').sum()} adolescents, "
      f"{(subj.cohort == 'adult').sum()} adults), {len(_a)} sessions, {len(analysed):,} responses = the LVL_0 handoff")

analysed sample: 282 participants (119 adolescents, 163 adults), 553 sessions, 552,281 responses = the LVL_0 handoff


In [4]:
# [cell 4]
# =============================================================================
# SAMPLE FLOW AND SINGLE-SESSION PARTICIPANTS
# =============================================================================
_all = dset.drop_duplicates("id")[["id", "age", "condition"]].assign(cohort=lambda d: cohort(d.age))
_old = _all[_all.age > P0["max_age"]]
_ses_all = dset.drop_duplicates(["id", "session"])
export("flow", pd.DataFrame([
    {"Stage": "dataset (all_subjects_long.csv)", "Participants": len(_all), "Sessions": len(_ses_all),
     "Adolescents": int((_all.cohort == "adolescent").sum()), "Adults": int((_all.cohort == "adult").sum()),
     "Note": ""},
    {"Stage": f"excluded: older than {P0['max_age']}", "Participants": len(_old),
     "Sessions": int(_ses_all.id.isin(_old.id).sum()), "Adolescents": int((_old.cohort == "adolescent").sum()),
     "Adults": int((_old.cohort == "adult").sum()),
     "Note": f"ages {_old.age.min():.0f}-{_old.age.max():.0f}; M = {_old.age.mean():.2f}, SD = {_old.age.std():.2f}"},
    {"Stage": "analysed sample (LVL_0 handoff)", "Participants": len(subj), "Sessions": len(_a),
     "Adolescents": int((subj.cohort == "adolescent").sum()), "Adults": int((subj.cohort == "adult").sum()),
     "Note": f"T2 {int((subj.condition == 'T2').sum())}, T3 {int((subj.condition == 'T3').sum())} participants"},
]), "Sample flow")
_single = subj[subj.n_sessions == 1][["id", "cohort", "condition"]].copy()
_single["session analysed"] = _single.id.map(_a.groupby("id").session.first())
_single["category"] = _single.id.map(lambda i: SESSION_MISSING_REASONS.get(i, ("no ledger entry", ""))[0])
_single["description"] = _single.id.map(lambda i: SESSION_MISSING_REASONS.get(i, ("", ""))[1])
export("single_session", _single.sort_values(["cohort", "id"]).reset_index(drop=True),
       "Participants with one analysed session, with the ledger's reason")
export("single_session_counts", _single.groupby(["cohort", "category"]).size().rename("n").reset_index(),
       "Single-session participants by cohort and reason")

Sample flow


,Stage,Participants,Sessions,Adolescents,Adults,Note
0,dataset (all_subjects_long.csv),293,572,119,174,
1,excluded: older than 30,11,19,0,11,"ages 31-56; M = 40.55, SD = 9.00"
2,analysed sample (LVL_0 handoff),282,553,119,163,"T2 139, T3 143 participants"


Participants with one analysed session, with the ledger's reason


,id,cohort,condition,session analysed,category,description
0,3032,adolescent,T2,2,wrong randomization,S1 run as T3 on a T2 ID
1,3046,adolescent,T2,1,incomplete session,S2 stopped after 17/200 trials; RA noted a pro...
2,4019,adolescent,T3,2,wrong randomization,S1 run as T2 on a T3 ID
3,4046,adolescent,T2,2,unknown,"S1 absent; unconfirmed, no record found"
4,1014,adult,T2,2,incomplete session,S1 stopped after 184/200 trials; no documentat...
5,1029,adult,T3,1,unknown,S2 never collected; no documentation identified
6,1071,adult,T3,2,unknown,"S1 absent; unconfirmed, no record found"
7,1078,adult,T2,1,procedural error,S1 administered a second time in place of S2
8,2078,adult,T2,1,did not return,not enrolled through SONA and did not return f...
9,2095,adult,T3,1,procedural error,S1 administered a second time in place of S2


Single-session participants by cohort and reason


,cohort,category,n
0,adolescent,incomplete session,1
1,adolescent,unknown,1
2,adolescent,wrong randomization,2
3,adult,did not return,2
4,adult,incomplete session,1
5,adult,procedural error,2
6,adult,unknown,2


In [5]:
# [cell 5]
# =============================================================================
# TABLE 0.1: SAMPLE BY CONDITION, T2 VS T3
# =============================================================================
# chi^2 tests without continuity correction; Mann-Whitney U two-sided. Sex as recorded in the dataset (female / male).
_pp = subj[["id", "condition", "sex", "age"]].assign(adolescent=lambda d: d.age < 18)
_ss = _a.merge(subj[["id", "condition"]], on="id")
_chi = lambda ct: stats.chi2_contingency(ct, correction=False)
rows = []
for lab, fn in (("Participants", lambda p, s: len(p)), ("Sessions, S1", lambda p, s: int((s.session == 1).sum())),
                ("Sessions, S2", lambda p, s: int((s.session == 2).sum())),
                ("Female, n (%)", lambda p, s: f"{int((p.sex == 'female').sum())} ({100 * (p.sex == 'female').mean():.1f})"),
                ("Adolescents (< 18), n (%)", lambda p, s: f"{int(p.adolescent.sum())} ({100 * p.adolescent.mean():.1f})"),
                ("Age, Mdn [Q1, Q3]", lambda p, s: f"{p.age.median():.0f} [{p.age.quantile(.25):.0f}, {p.age.quantile(.75):.0f}]")):
    r = {"Characteristic": lab}
    for cd in ("T2", "T3"):
        r[cd] = fn(_pp[_pp.condition == cd], _ss[_ss.condition == cd])
    r["Total"] = fn(_pp, _ss)
    rows.append(r)
_t_sex = _chi(pd.crosstab(_pp.condition, _pp.sex)); _t_ado = _chi(pd.crosstab(_pp.condition, _pp.adolescent))
_t_ses = _chi(pd.crosstab(_ss.condition, _ss.session))
_t_age = stats.mannwhitneyu(_pp.age[_pp.condition == "T2"], _pp.age[_pp.condition == "T3"])
_tests = {"Sessions, S1": ("χ²(1), sessions S1 / S2 × condition", _t_ses[0], _t_ses[1]),
          "Female, n (%)": ("χ²(1), sex × condition", _t_sex[0], _t_sex[1]),
          "Adolescents (< 18), n (%)": ("χ²(1), adolescent × condition", _t_ado[0], _t_ado[1]),
          "Age, Mdn [Q1, Q3]": ("Mann–Whitney U", _t_age.statistic, _t_age.pvalue)}
for r in rows:
    t = _tests.get(r["Characteristic"], ("", np.nan, np.nan))
    r.update({"T2 vs T3 test": t[0], "Statistic": t[1], "p": t[2]})
export("balance", pd.DataFrame(rows), "Table 0.1. Sample by condition: composition and T2 vs T3 comparisons")

Table 0.1. Sample by condition: composition and T2 vs T3 comparisons


,Characteristic,T2,T3,Total,T2 vs T3 test,Statistic,p
0,Participants,139,143,282,,NaN,NaN
1,"Sessions, S1",136,141,277,"χ²(1), sessions S1 / S2 × condition",0.001750,0.966628
2,"Sessions, S2",136,140,276,,NaN,NaN
3,"Female, n (%)",81 (58.3),83 (58.0),164 (58.2),"χ²(1), sex × condition",0.001551,0.968582
4,"Adolescents (< 18), n (%)",59 (42.4),60 (42.0),119 (42.2),"χ²(1), adolescent × condition",0.006882,0.933886
5,"Age, Mdn [Q1, Q3]","19 [14, 21]","19 [15, 21]","19 [14, 21]",Mann–Whitney U,9890.500000,0.944493


In [6]:
# [cell 6]
# =============================================================================
# SAMPLE CHARACTERISTICS BY COHORT, CONDITION AND SEX (the manuscript's demographics table)
# =============================================================================
# One row per participant. n, % of the analysed sample, M (SD) of age and of the depression and anxiety T-scores, and
# the interval between sessions (IBS, hours): median and MAD (unscaled: median |x - median|) among participants with
# both sessions.
N_SAMPLE = len(subj)
_mad = lambda x: (x.dropna() - x.dropna().median()).abs().median() if x.notna().any() else np.nan


def summarize(g, label):
    return {"Group": label, "n": len(g), "%": 100 * len(g) / N_SAMPLE,
            "Age M": g.age.mean(), "Age SD": g.age.std(), "Depression M": g.dep_t.mean(), "Depression SD": g.dep_t.std(),
            "Anxiety M": g.anx_t.mean(), "Anxiety SD": g.anx_t.std(),
            "IBS (hours) Mdn": g.S1_S2_hours.median(), "IBS (hours) MAD (unscaled)": _mad(g.S1_S2_hours)}


rows = [summarize(subj[subj.cohort == c], c.capitalize()) for c in ("adolescent", "adult")] + [summarize(subj, "Overall")]
rows += [summarize(subj[(subj.cohort == c) & (subj.condition == t)], f"{c.capitalize()}, {t}")
         for c in ("adolescent", "adult") for t in ("T2", "T3")]
rows += [summarize(subj[subj.condition == t], f"Overall, {t}") for t in ("T2", "T3")]
rows += [summarize(subj[(subj.cohort == c) & (subj.sex == s)], f"{c.capitalize()}, {s}")
         for c in ("adolescent", "adult") for s in ("female", "male")]
rows += [summarize(subj[subj.sex == s], f"Overall, {s}") for s in ("female", "male")]
export("table", pd.DataFrame(rows), "Sample characteristics by cohort, time point and sex")

Sample characteristics by cohort, time point and sex


,Group,n,%,Age M,Age SD,Depression M,Depression SD,Anxiety M,Anxiety SD,IBS (hours) Mdn,IBS (hours) MAD (unscaled)
0,Adolescent,119,42.198582,13.680672,2.265933,52.766071,7.276218,48.631579,9.116797,1.200000,0.266667
1,Adult,163,57.801418,21.110429,2.643431,52.135220,8.487445,57.420988,8.798090,23.625000,22.691667
2,Overall,282,100.000000,17.975177,4.438118,52.395941,8.001080,53.790580,9.913043,1.516667,0.766667
3,"Adolescent, T2",59,20.921986,13.576271,2.159210,53.335714,7.107497,49.700000,9.242845,1.233333,0.266667
4,"Adolescent, T3",60,21.276596,13.783333,2.379942,52.196429,7.461366,47.600000,8.951918,1.183333,0.300000
5,"Adult, T2",80,28.368794,21.337500,2.984957,50.698701,8.300579,56.215000,8.387998,23.200000,22.150000
6,"Adult, T3",83,29.432624,20.891566,2.263238,53.484146,8.489157,58.597561,9.077335,23.716667,22.850000
7,"Overall, T2",139,49.290780,18.043165,4.677839,51.809023,7.901011,53.532353,9.291644,1.650000,0.950000
8,"Overall, T3",143,50.709220,17.909091,4.207487,52.961594,8.084619,54.041429,10.508907,1.408333,0.658333
9,"Adolescent, female",69,24.468085,13.913043,2.240924,53.482813,7.800052,48.838462,9.391830,1.216667,0.283333


In [7]:
# [cell 7]
# =============================================================================
# SESSION INTERVALS
# =============================================================================
# Among participants with both sessions: same calendar day; different days (days apart = interval in hours / 24).
# More than 7 days apart: n and % of the cohort.
rows = []
for c, g in [("Adolescent", subj[subj.cohort == "adolescent"]), ("Adult", subj[subj.cohort == "adult"]), ("Overall", subj)]:
    two = g[g.n_sessions == 2]
    diff = two[two.same_day == 0]
    rows.append({"Cohort": c, "Participants": len(g), "Both sessions": len(two),
                 "Same day, n": int((two.same_day == 1).sum()), "Same day, %": 100 * (two.same_day == 1).mean(),
                 "Different days, n": len(diff), "Days apart M": (diff.S1_S2_hours / 24).mean(),
                 "Days apart SD": (diff.S1_S2_hours / 24).std(),
                 "More than 7 days apart, n": int((g.S1_S2_hours > 7 * 24).sum()),
                 "More than 7 days apart, % of cohort": 100 * (g.S1_S2_hours > 7 * 24).sum() / len(g)})
export("intervals", pd.DataFrame(rows), "Interval between sessions")

Interval between sessions


,Cohort,Participants,Both sessions,"Same day, n","Same day, %","Different days, n",Days apart M,Days apart SD,"More than 7 days apart, n","More than 7 days apart, % of cohort"
0,Adolescent,119,115,103,89.565217,12,7.491782,5.347895,7,5.882353
1,Adult,163,156,71,45.512821,85,3.536201,4.036780,9,5.521472
2,Overall,282,271,174,64.206642,97,4.025551,4.387436,16,5.673759


In [8]:
# [cell 8]
# =============================================================================
# OUT-OF-RANGE RESPONSES, AND THE MANUSCRIPT COUNTS WITHOUT A RECORD
# =============================================================================
# Valid response categories of each scale (the task's scales; LVL_1_CONFIG's scale_ranges). The original task version
# accepted keys 1-9 on every scale; responses outside a scale's range are counted here and stay in the dataset (LVL_1
# sets them to missing for the rating-based features).
VALID_RANGE = {"valence": (1, 9), "arousal": (1, 9), "emotion": (1, 7), "similarity": (1, 6), "threat": (1, 5)}
_r = analysed[["id", "scale", "response"]].copy()
_r["oor"] = ((_r.response < _r.scale.map(lambda s: VALID_RANGE[s][0]))
             | (_r.response > _r.scale.map(lambda s: VALID_RANGE[s][1])))
rows = [{"Scale": s, "Valid": f"{lo}-{hi}", "Responses": int((_r.scale == s).sum()),
         "Out of range, n": int(_r.oor[_r.scale == s].sum()), "Out of range, %": 100 * _r.oor[_r.scale == s].mean(),
         "Participants with any": int(_r[_r.oor & (_r.scale == s)].id.nunique())} for s, (lo, hi) in VALID_RANGE.items()]
rows.append({"Scale": "all scales", "Valid": "", "Responses": len(_r), "Out of range, n": int(_r.oor.sum()),
             "Out of range, %": 100 * _r.oor.mean(), "Participants with any": int(_r[_r.oor].id.nunique())})
export("out_of_range", pd.DataFrame(rows), "Out-of-range responses in the analysed sample")
print("\nManuscript counts without a record on this machine (flags; not computed):")
for what, needs in UNSUPPORTED:
    print(f"  - {what}  [needs {needs}]")

Out-of-range responses in the analysed sample


,Scale,Valid,Responses,"Out of range, n","Out of range, %",Participants with any
0,valence,1-9,110458,0,0.000000,0
1,arousal,1-9,110459,0,0.000000,0
2,emotion,1-7,110462,482,0.436349,98
3,similarity,1-6,110448,209,0.189229,46
4,threat,1-5,110454,386,0.349467,71
5,all scales,,552281,1077,0.195009,133



Manuscript counts without a record on this machine (flags; not computed):
  - enrolled: adults 280, adolescents 120 (400)  [needs enrolment records]
  - adult males compensated monetarily: 31  [needs compensation records]
  - adults in the pilot task: 59  [needs pilot records]
  - sessions that ought to have been collected: 600  [needs enrolment records]
  - participants on the original task version: 264 (93.6%)  [needs session dates (the *_w_rts.tsv file names)]
